#Build Results Fact

- 1-Ler tabela silver results
- 2-Ler tabela silver sprints
- 3-Adicionar nova coluna session_type com valores RACE ou SPRINT
- 4-Fazer UNION entre results e sprints
- 5-Derivar colunas adicionais
- > is_win -> Indica que o piloto venceu a corrida
- > is_podium -> Indica que o piloto ficou no pódio (1º, 2º ou 3º)
- > has_points -> Indica que o piloto pontuou
- 6-Escrever os dados transformados na tabela gold fact_session_results

In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/04.gold-helpers

In [0]:
target_table = f"{catalog_name}.{gold_schema}.fact_session_results"

In [0]:
from pyspark.sql import functions as F

In [0]:
results_df = (
    spark.table(f"{catalog_name}.{silver_schema}.results")
        .filter((F.col("batch_id") == v_batch_id))
        .withColumn("session_type", F.lit("RACE"))
        .drop("race_name", "race_date", "ingestion_timestamp", "source_file", "batch_id", "created_timestamp", "update_timestamp")
    )


In [0]:
sprints_df = (
    spark.table(f"{catalog_name}.{silver_schema}.sprints")
        .filter((F.col("batch_id") == v_batch_id))
        .withColumn("session_type", F.lit("SPRINT"))
        .drop("race_name", "race_date", "ingestion_timestamp", "source_file","batch_id", "created_timestamp", "update_timestamp")
    )


In [0]:
results_sprints_df = results_df.unionByName(sprints_df)

In [0]:
fact_session_results_df = (
    results_sprints_df
        .withColumn("is_win", F.col("finish_position") == 1) #ai  ser true se a condição for satisfeita
        .withColumn("is_podium", F.col("finish_position").between(1,3)) #o valor entre 1,2 e 3 vai ser true
        .withColumn("has_point", F.col("points") > 0) #se o valor for maior que zero sera true
    
)

In [0]:
write_to_gold(
    input_df=fact_session_results_df,
    target_table=target_table,
    merge_condition="""
        t.season = s.season
        AND t.round = s.round
        AND t.constructor_id = s.constructor_id
        AND t.driver_id = s.driver_id
        AND t.session_type = s.session_type
    """,
      columns_to_update=[
      "grid_position",
      "completed_laps",
      "car_number",
      "points",
      "finish_position",
      "finish_position_text",
      "status",
      "is_win",
      "is_podium",
      "has_point"
  ]
)

In [0]:
display(spark.table(target_table))